# Predicting 30 Day Repeat Purchase

### Customer analytics and machine learning project

This project predicts whether an existing customer will purchase again within 30 days using historical shopping, loyalty and campaign features.

I approached the problem as an analyst first. The aim is not simply to get the highest classification score. The model should help a retention team prioritize customers while making the precision and recall tradeoff visible.

The modelling work compares:

* majority class baseline
* Logistic Regression
* constrained Decision Tree
* Random Forest
* threshold tuning
* final holdout evaluation
* coefficient interpretation
* model packaging

This is a prediction project. Historical campaign exposure can be predictive without proving that the campaign caused a repeat purchase.


## 1. Load the customer modelling table

The customer level modelling table contains 1,725 customers. The target is `repeat_purchase_30d`, where 1 means the customer made another purchase within 30 days after the cutoff.

I exclude identifiers and a few redundant engineered fields before modelling.


In [ ]:
import pandas as pd
import numpy as np

customer_model = pd.read_csv("loyalty_customer_model.csv")

excluded_columns = [
    "customer_id",
    "repeat_purchase_30d",
    "unique_categories",
    "cross_shopper",
    "discounted_orders"
]

X = customer_model.drop(columns=excluded_columns).copy()
y = customer_model["repeat_purchase_30d"].copy()

print("Feature shape:", X.shape)
print("Target shape:", y.shape)


Feature shape: (1725, 12)
Target shape: (1725,)


## 2. Train and test split

Repeat buyers make up about 30% of the data, so I use a stratified split to preserve the class balance in both sets.

The test set is kept untouched until the final model and threshold are selected.


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=43
)

print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Training repeat rate (%):", round(y_train.mean() * 100, 2))
print("Test repeat rate (%):", round(y_test.mean() * 100, 2))


Training shape: (1380, 12)
Test shape: (345, 12)
Training repeat rate (%): 29.64
Test repeat rate (%): 29.57


## 3. Preprocessing and baseline

Numeric features are standardized for Logistic Regression. Loyalty tier is one hot encoded.

I also create a majority class baseline. This is important because accuracy alone can look acceptable even when a model never identifies a repeat buyer.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

categorical_features = ["loyalty_tier_at_cutoff"]
numeric_features = [
    column for column in X_train.columns
    if column not in categorical_features
]

preprocessor = ColumnTransformer([
    ("numeric", StandardScaler(), numeric_features),
    (
        "categorical",
        OneHotEncoder(handle_unknown="ignore"),
        categorical_features
    )
])

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=43
)

baseline = DummyClassifier(strategy="most_frequent")

baseline_scores = cross_validate(
    baseline,
    X_train,
    y_train,
    cv=cv,
    scoring={
        "accuracy": "accuracy",
        "recall": "recall",
        "roc_auc": "roc_auc"
    }
)

baseline_summary = pd.Series({
    "accuracy": baseline_scores["test_accuracy"].mean(),
    "recall": baseline_scores["test_recall"].mean(),
    "roc_auc": baseline_scores["test_roc_auc"].mean()
})

print(baseline_summary)


accuracy    0.703623
recall      0.000000
roc_auc     0.500000
dtype: float64


The majority baseline gets about 70% accuracy but zero recall for repeat buyers. That makes it a poor business model despite the apparently reasonable accuracy.


## 4. Logistic Regression

I start with Logistic Regression because it gives an interpretable probability model and a strong benchmark for a binary classification problem.


In [ ]:
logistic_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

metrics = ["accuracy", "precision", "recall", "f1", "roc_auc"]

logistic_scores = cross_validate(
    estimator=logistic_pipeline,
    X=X_train,
    y=y_train,
    cv=cv,
    scoring=metrics,
    return_train_score=True
)

logistic_summary = pd.DataFrame(
    index=metrics,
    columns=["training", "validation"],
    dtype=float
)

for metric in metrics:
    logistic_summary.loc[metric, "training"] = (
        logistic_scores["train_" + metric].mean()
    )
    logistic_summary.loc[metric, "validation"] = (
        logistic_scores["test_" + metric].mean()
    )

print(logistic_summary.round(3))


           training  validation
accuracy      0.708       0.704
precision     0.523       0.514
recall        0.160       0.157
f1            0.244       0.239
roc_auc       0.676       0.658


The validation ROC AUC is 0.658. At the default classification threshold, recall is low, which matters because the business objective is to identify more likely repeat buyers for prioritization.


## 5. Decision Tree comparison

I compare Logistic Regression with a deliberately constrained Decision Tree. The tree is kept shallow and each leaf must contain at least 30 training customers to reduce overfitting.


In [ ]:
from sklearn.tree import DecisionTreeClassifier

tree_preprocessor = ColumnTransformer([
    ("numeric", "passthrough", numeric_features),
    (
        "categorical",
        OneHotEncoder(handle_unknown="ignore"),
        categorical_features
    )
])

tree_pipeline = Pipeline([
    ("preprocessor", tree_preprocessor),
    (
        "model",
        DecisionTreeClassifier(
            max_depth=3,
            min_samples_leaf=30,
            random_state=42
        )
    )
])

tree_scores = cross_validate(
    estimator=tree_pipeline,
    X=X_train,
    y=y_train,
    cv=cv,
    scoring=["roc_auc", "precision", "recall"],
    return_train_score=True
)

print("Training AUC:", round(tree_scores["train_roc_auc"].mean(), 3))
print("Validation AUC:", round(tree_scores["test_roc_auc"].mean(), 3))
print("Validation precision:", round(tree_scores["test_precision"].mean(), 3))
print("Validation recall:", round(tree_scores["test_recall"].mean(), 3))


Training AUC: 0.662
Validation AUC: 0.611
Validation precision: 0.507
Validation recall: 0.142


The constrained tree is easier to explain visually, but its validation ROC AUC is below Logistic Regression.


## 6. Random Forest comparison

I also test Random Forest to see whether a more flexible ensemble captures useful nonlinear patterns.

The forest is still constrained with shallow trees and a minimum leaf size. I am not doing a large hyperparameter search here because the goal is model comparison and learning, not squeezing every possible point from the sample.


In [ ]:
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier

forest_pipeline = Pipeline([
    ("preprocessor", clone(tree_preprocessor)),
    (
        "model",
        RandomForestClassifier(
            n_estimators=100,
            max_depth=5,
            min_samples_leaf=10,
            max_features="sqrt",
            bootstrap=True,
            random_state=42,
            n_jobs=-1
        )
    )
])

forest_scores = cross_validate(
    estimator=forest_pipeline,
    X=X_train,
    y=y_train,
    cv=cv,
    scoring=["roc_auc", "precision", "recall"],
    return_train_score=True
)

print("Training AUC:", round(forest_scores["train_roc_auc"].mean(), 3))
print("Validation AUC:", round(forest_scores["test_roc_auc"].mean(), 3))
print("Validation precision:", round(forest_scores["test_precision"].mean(), 3))
print("Validation recall:", round(forest_scores["test_recall"].mean(), 3))


Training AUC: 0.78
Validation AUC: 0.659
Validation precision: 0.536
Validation recall: 0.118


### Model selection

Random Forest and Logistic Regression are essentially tied on validation ROC AUC at 0.659 and 0.658.

Random Forest has a noticeably larger training to validation gap, while Logistic Regression is easier to interpret and produces almost the same validation ranking performance.

For this version of the project I therefore keep Logistic Regression as the final model. I would revisit the ensemble if later time based validation showed a consistent improvement.


## 7. Threshold tuning

The default 0.50 threshold is conservative. I use out of fold probabilities from the training set and compare several thresholds.

This keeps threshold selection separate from the final test set.


In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import precision_score, recall_score, f1_score

validation_probabilities = cross_val_predict(
    estimator=logistic_pipeline,
    X=X_train,
    y=y_train,
    cv=cv,
    method="predict_proba"
)[:, 1]

threshold_results = []

for threshold in [0.20, 0.25, 0.30, 0.40, 0.50]:
    predictions = (
        validation_probabilities >= threshold
    ).astype(int)

    threshold_results.append({
        "threshold": threshold,
        "precision": precision_score(
            y_train,
            predictions,
            zero_division=0
        ),
        "recall": recall_score(y_train, predictions),
        "f1": f1_score(y_train, predictions)
    })

threshold_comparison = pd.DataFrame(threshold_results)
print(threshold_comparison.round(3).to_string(index=False))


 threshold  precision  recall    f1
      0.20      0.336   0.846 0.481
      0.25      0.393   0.726 0.510
      0.30      0.419   0.567 0.482
      0.40      0.477   0.308 0.374
      0.50      0.504   0.156 0.239


Among the thresholds tested, 0.25 gives the highest validation F1. It also raises recall substantially compared with 0.50.

That does not make 0.25 universally optimal. A production threshold should depend on contact cost, expected margin, campaign capacity and the cost of false positives.


## 8. Final holdout evaluation

The model choice and threshold are now fixed. I fit Logistic Regression on all training customers and evaluate once on the untouched test set.


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

selected_model = logistic_pipeline
selected_threshold = 0.25

selected_model.fit(X_train, y_train)

test_probabilities = selected_model.predict_proba(X_test)[:, 1]
test_predictions = (
    test_probabilities >= selected_threshold
).astype(int)

test_results = pd.Series({
    "accuracy": accuracy_score(y_test, test_predictions),
    "precision": precision_score(
        y_test,
        test_predictions,
        zero_division=0
    ),
    "recall": recall_score(y_test, test_predictions),
    "f1": f1_score(y_test, test_predictions),
    "roc_auc": roc_auc_score(y_test, test_probabilities)
})

print(test_results.round(3))
print("Confusion matrix:")
print(confusion_matrix(y_test, test_predictions))


accuracy     0.536
precision    0.355
recall       0.696
f1           0.470
roc_auc      0.605
dtype: float64
Confusion matrix:
[[114 129]
 [ 31  71]]


### What the holdout result means

At the 0.25 threshold:

* precision is 35.5%, so 71 of 200 flagged customers actually repeated
* recall is 69.6%, so the model identified 71 of 102 actual repeat buyers
* F1 is 0.470
* ROC AUC is 0.605

The lower threshold captures more actual repeat buyers, but it also creates many false positives. I would use this model as a prioritization layer for inexpensive outreach, not as a reason to automatically give discounts.

The drop from validation ROC AUC to 0.605 on the holdout also shows that the model still has limited ranking power.


## 9. Interpreting Logistic Regression coefficients

Because the numeric variables were standardized, the numeric coefficients can be compared directionally inside this fitted model.

The coefficients describe association with the modelled probability. They should not be interpreted as causal effects.


In [ ]:
feature_names = (
    selected_model.named_steps["preprocessor"]
    .get_feature_names_out()
)

coefficients = (
    selected_model.named_steps["model"]
    .coef_[0]
)

coefficient_table = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients
})

print(
    coefficient_table
    .sort_values("coefficient", ascending=False)
    .round(3)
    .to_string(index=False)
)


                                      feature  coefficient
                           numeric__frequency        0.443
    categorical__loyalty_tier_at_cutoff_Gold        0.255
            numeric__points_balance_at_cutoff        0.196
           numeric__campaign_assignment_count        0.144
                     numeric__points_redeemer        0.099
              numeric__discounted_order_share        0.073
                     numeric__avg_order_value        0.038
                        numeric__recency_days        0.023
  categorical__loyalty_tier_at_cutoff_Silver       -0.002
                 numeric__digital_order_share       -0.010
categorical__loyalty_tier_at_cutoff_Platinum       -0.013
                       numeric__unique_brands       -0.024
                         numeric__total_spend       -0.092
                         numeric__tenure_days       -0.194
  categorical__loyalty_tier_at_cutoff_Bronze       -0.242


Frequency has the strongest positive coefficient in this run. Gold tier, points balance and historical campaign assignment count are also positively associated with predicted repeat purchase probability.

The campaign coefficient is especially important to interpret carefully. It can help prediction, but it does not show that additional campaign exposure caused customers to buy again.


## 10. Save the model

The saved package includes the fitted preprocessing pipeline, Logistic Regression model, selected probability threshold and expected input columns.


In [ ]:
import joblib

model_package = {
    "pipeline": selected_model,
    "threshold": selected_threshold,
    "feature_columns": X_train.columns.tolist()
}

joblib.dump(
    model_package,
    "loyalty_repeat_purchase_model.joblib"
)

print("Model saved.")


Model saved.


In [ ]:
loaded_package = joblib.load(
    "loyalty_repeat_purchase_model.joblib"
)

loaded_model = loaded_package["pipeline"]

loaded_probabilities = loaded_model.predict_proba(
    X_test[loaded_package["feature_columns"]]
)[:, 1]

print(
    "Predictions match:",
    np.allclose(test_probabilities, loaded_probabilities)
)


Predictions match: True


## 11. Business conclusion

The model can help rank existing customers for retention activity, but its current performance is not strong enough for automated targeting.

Logistic Regression was selected because it delivered almost the same validation ROC AUC as Random Forest with a smaller training to validation gap and clearer interpretation.

A 0.25 threshold improved recall and produced the best F1 among the thresholds tested, but the holdout results make the tradeoff clear: more repeat buyers are found at the cost of many false positives.

### Limitations

* the dataset is synthetic
* test ROC AUC is modest at 0.605
* the evaluation uses one customer split rather than a later time period
* historical campaign exposure is predictive, not causal evidence
* the model should be validated on future periods before any real deployment

### What I would improve next

* use a later time window for validation
* compare calibration across models
* test a smaller feature set for stability
* connect threshold selection to campaign cost and expected margin
* monitor performance drift if the model is used repeatedly
